# Neural Network with Lag12 Feature

This notebook explores the addition of 12-month lagged delay rate, `delay_rate_lag12`, to the nowcasting model using neural network. This is motivated by previous findings in notebooks 15a and 16, where it was demonstrated that `delay_rate_lag12` (same month last year) generally improves performance, especially for the regression models.

For fair comparison, all models (Ridge, RF, Logistic, XGBoost, NN) will be trained on the same dataset, both with and without lag12.

## Reference (notebook 11, without lag12)

| Model | Test R² | Test RMSE | Test F1 |
|-------|---------|-----------|--------|
| Ridge | 0.455 | 0.085 | - |
| Random Forest | 0.478 | 0.083 | 0.694 |
| Neural Network | 0.489 | 0.082 | 0.737 |
| XGBoost | - | - | 0.725 |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, f1_score, roc_auc_score, precision_score, recall_score
)
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.linewidth'] = 1.5

try:
    import xgboost as xgb
    HAS_XGB = True
    print("XGBoost available")
except ImportError:
    HAS_XGB = False
    print("XGBoost not installed")

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    HAS_TF = True
    tf.random.set_seed(42)
    print("TensorFlow {} available".format(tf.__version__))
except ImportError:
    HAS_TF = False
    print("TensorFlow not installed. Neural network sections will be skipped.")

np.random.seed(42)

%matplotlib inline

## 1. Data Preparation

Identical filtering to notebook 11.

In [ ]:
df = pd.read_csv('../data/processed/ml_training_data_multiroute_hols.csv')

df['year_month_dt'] = pd.to_datetime(df['year_month'])
df['month_num'] = df['year_month_dt'].dt.month
df['year'] = df['year'].astype(int)
df['airline_route'] = df['airline'] + '_' + df['departing_port'] + '_' + df['arriving_port']
df['route'] = df['departing_port'] + '_' + df['arriving_port']
df = df.sort_values(['airline_route', 'year_month_dt']).reset_index(drop=True)

print("Original shape: {}".format(df.shape))
print("Date range: {} to {}".format(df['year_month'].min(), df['year_month'].max()))

In [ ]:
# Filter low-volume airline-routes (same threshold as notebook 11)
volume_threshold = 50
airline_route_volume = df.groupby('airline_route')['sectors_scheduled'].mean().reset_index()
airline_route_volume.columns = ['airline_route', 'avg_volume']
high_volume_ar = airline_route_volume[airline_route_volume['avg_volume'] >= volume_threshold]['airline_route'].tolist()

excluded = airline_route_volume[airline_route_volume['avg_volume'] < volume_threshold].sort_values('avg_volume')
print("Volume threshold: {} flights/month".format(volume_threshold))
print("Excluded airline-routes: {}".format(len(excluded)))

df_filtered = df[df['airline_route'].isin(high_volume_ar)].copy()
print("\nRecords before filtering: {}".format(len(df)))
print("Records after filtering:  {}".format(len(df_filtered)))

In [ ]:
# Exclude anomalous routes (same as notebook 11)
anomalous_routes = ['Melbourne_Hobart', 'Adelaide_Sydney']
records_before = len(df_filtered)
df_filtered = df_filtered[~df_filtered['route'].isin(anomalous_routes)].copy()
print("Excluded anomalous routes: {}".format(anomalous_routes))
print("Records before: {}".format(records_before))
print("Records after:  {}".format(len(df_filtered)))

## 2. Feature Engineering

In [ ]:
# Lagged features (same as notebook 11)
df_filtered['delay_rate_lag1'] = df_filtered.groupby('airline_route')['delay_rate'].shift(1)
df_filtered['delay_rate_lag2'] = df_filtered.groupby('airline_route')['delay_rate'].shift(2)
df_filtered['delay_rate_gradient'] = df_filtered['delay_rate_lag1'] - df_filtered['delay_rate_lag2']

# NEW: lag12 (same month last year)
df_filtered['delay_rate_lag12'] = df_filtered.groupby('airline_route')['delay_rate'].shift(12)

# Cyclical month encoding
df_filtered['month_sin'] = np.sin(2 * np.pi * df_filtered['month_num'] / 12)
df_filtered['month_cos'] = np.cos(2 * np.pi * df_filtered['month_num'] / 12)

# One-hot encoding
airline_dummies = pd.get_dummies(df_filtered['airline'], prefix='airline')
df_filtered = pd.concat([df_filtered, airline_dummies], axis=1)
airline_cols = list(airline_dummies.columns)

route_dummies = pd.get_dummies(df_filtered['route'], prefix='route')
df_filtered = pd.concat([df_filtered, route_dummies], axis=1)
route_cols = list(route_dummies.columns)

# Weather features with transformations (same as notebook 11)
df_filtered['rainy_days_arr_exp'] = np.exp(df_filtered['rainy_days_arr'] / df_filtered['rainy_days_arr'].max())
df_filtered['temp_volatility_total'] = df_filtered['temp_volatility_dep'] + df_filtered['temp_volatility_arr']
df_filtered['temp_volatility_total_exp'] = np.exp(df_filtered['temp_volatility_total'] / df_filtered['temp_volatility_total'].max())
df_filtered['extreme_weather_days_total'] = df_filtered['extreme_weather_days_dep'] + df_filtered['extreme_weather_days_arr']

print("Airlines: {}".format(len(airline_cols)))
print("Routes:   {}".format(len(route_cols)))

## 3. Data Loss from Lag12

In [ ]:
df_baseline = df_filtered.dropna(subset=['delay_rate_lag1', 'delay_rate_lag2', 'delay_rate_gradient']).copy()
df_with_lag12 = df_filtered.dropna(subset=['delay_rate_lag1', 'delay_rate_lag2', 'delay_rate_gradient', 'delay_rate_lag12']).copy()

print("Data availability:")
print("  Baseline (no lag12): {:,} rows".format(len(df_baseline)))
print("  With lag12:          {:,} rows".format(len(df_with_lag12)))
print("  Rows lost:           {:,} ({:.1f}%)".format(
    len(df_baseline) - len(df_with_lag12),
    (len(df_baseline) - len(df_with_lag12)) / len(df_baseline) * 100
))
print()
print("Using lag12-available subset for both baseline and lag12 models (fair comparison).")

## 4. Train/Validation/Test Split

In [ ]:
# Use lag12-available subset for fair comparison
df_clean = df_with_lag12.copy()

train_mask = (((df_clean['year'] >= 2010) & (df_clean['year'] <= 2017)) | (df_clean['year'] == 2023))
val_mask = ((df_clean['year'] == 2018) | (df_clean['year'] == 2024))
test_mask = ((df_clean['year'] == 2019) | (df_clean['year'] >= 2025))

print("Train (2010-2017, 2023): {:,} samples".format(train_mask.sum()))
print("Validation (2018, 2024): {:,} samples".format(val_mask.sum()))
print("Test (2019, 2025+):      {:,} samples".format(test_mask.sum()))

In [ ]:
# Feature sets
base_features = airline_cols + route_cols + ['month_sin', 'month_cos', 'delay_rate_lag1', 'sectors_scheduled']
weather_features = ['rainy_days_arr_exp', 'delay_rate_gradient', 'temp_volatility_total_exp', 'extreme_weather_days_total']
holiday_features = ['n_public_holidays_total', 'pct_school_holiday']

# Baseline: same as notebook 11
FEATURES_BASELINE = base_features + weather_features + holiday_features

# With lag12
FEATURES_LAG12 = base_features + weather_features + ['delay_rate_lag12'] + holiday_features

print("Baseline features: {}".format(len(FEATURES_BASELINE)))
print("With lag12:        {}".format(len(FEATURES_LAG12)))
print()
print("Feature breakdown:")
print("  Airline encoding: {}".format(len(airline_cols)))
print("  Route encoding:   {}".format(len(route_cols)))
print("  Month encoding:   2 (sin, cos)")
print("  Lag features:     2 (lag1, gradient) + lag12")
print("  Volume:           1 (sectors_scheduled)")
print("  Weather:          3 (rainy, temp_volatility, extreme)")
print("  Holidays:         2 (public, school)")

## 5. Helper Functions

In [ ]:
def build_nn(input_dim, hidden1=32, hidden2=16, dropout=0.3, output_activation='linear'):
    """Build a feedforward neural network."""
    model = keras.Sequential([
        layers.Dense(hidden1, activation='relu', input_shape=(input_dim,)),
        layers.Dropout(dropout),
        layers.Dense(hidden2, activation='relu'),
        layers.Dropout(dropout),
        layers.Dense(1, activation=output_activation)
    ])
    return model


def train_traditional_models(df, features, train_mask, val_mask, test_mask, label):
    """Train Ridge, RF, Logistic, RF-clf, XGBoost and return results dict."""
    X_train = df.loc[train_mask, features].values
    X_val = df.loc[val_mask, features].values
    X_test = df.loc[test_mask, features].values

    y_train = df.loc[train_mask, 'delay_rate'].values
    y_val = df.loc[val_mask, 'delay_rate'].values
    y_test = df.loc[test_mask, 'delay_rate'].values

    y_train_clf = df.loc[train_mask, 'is_high_delay'].values
    y_val_clf = df.loc[val_mask, 'is_high_delay'].values
    y_test_clf = df.loc[test_mask, 'is_high_delay'].values

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    results = {'label': label}

    # Ridge
    ridge = Ridge(alpha=1.0)
    ridge.fit(X_train_scaled, y_train)
    ridge_pred = ridge.predict(X_test_scaled)
    results['ridge_r2'] = r2_score(y_test, ridge_pred)
    results['ridge_rmse'] = np.sqrt(mean_squared_error(y_test, ridge_pred))

    # RF Regression
    rf = RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    rf_pred = rf.predict(X_test)
    results['rf_r2'] = r2_score(y_test, rf_pred)
    results['rf_rmse'] = np.sqrt(mean_squared_error(y_test, rf_pred))
    results['rf_model'] = rf

    # Logistic Regression
    logreg = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
    logreg.fit(X_train_scaled, y_train_clf)
    logreg_pred = logreg.predict(X_test_scaled)
    results['logreg_f1'] = f1_score(y_test_clf, logreg_pred)

    # RF Classification
    rf_clf = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
    rf_clf.fit(X_train, y_train_clf)
    rf_clf_pred = rf_clf.predict(X_test)
    results['rf_clf_f1'] = f1_score(y_test_clf, rf_clf_pred)

    # XGBoost Classification
    if HAS_XGB:
        xgb_clf = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1,
                                     min_child_weight=5, random_state=42, n_jobs=-1)
        xgb_clf.fit(X_train, y_train_clf, eval_set=[(X_val, y_val_clf)], verbose=False)
        xgb_pred = xgb_clf.predict(X_test)
        results['xgb_f1'] = f1_score(y_test_clf, xgb_pred)
    else:
        results['xgb_f1'] = np.nan

    # Store data arrays for NN training
    results['X_train'] = X_train
    results['X_val'] = X_val
    results['X_test'] = X_test
    results['X_train_scaled'] = X_train_scaled
    results['X_val_scaled'] = X_val_scaled
    results['X_test_scaled'] = X_test_scaled
    results['y_train'] = y_train
    results['y_val'] = y_val
    results['y_test'] = y_test
    results['y_train_clf'] = y_train_clf
    results['y_val_clf'] = y_val_clf
    results['y_test_clf'] = y_test_clf

    return results


def train_nn_regression(X_train_scaled, y_train, X_val_scaled, y_val, X_test_scaled, y_test):
    """Train NN for regression and return results."""
    tf.random.set_seed(42)
    np.random.seed(42)

    nn = build_nn(input_dim=X_train_scaled.shape[1], output_activation='linear')
    nn.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=20, restore_best_weights=True, verbose=1
    )

    history = nn.fit(
        X_train_scaled, y_train,
        validation_data=(X_val_scaled, y_val),
        epochs=500, batch_size=64,
        callbacks=[early_stop],
        verbose=0
    )

    best_epoch = np.argmin(history.history['val_loss']) + 1
    nn_pred = nn.predict(X_test_scaled, verbose=0).flatten()

    return {
        'model': nn,
        'history': history,
        'best_epoch': best_epoch,
        'predictions': nn_pred,
        'r2': r2_score(y_test, nn_pred),
        'rmse': np.sqrt(mean_squared_error(y_test, nn_pred)),
        'mae': mean_absolute_error(y_test, nn_pred),
    }


def train_nn_classification(X_train_scaled, y_train_clf, X_val_scaled, y_val_clf,
                            X_test_scaled, y_test_clf):
    """Train NN for classification and return results."""
    tf.random.set_seed(42)
    np.random.seed(42)

    nn = build_nn(input_dim=X_train_scaled.shape[1], output_activation='sigmoid')
    nn.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
               loss='binary_crossentropy', metrics=['accuracy'])

    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=20, restore_best_weights=True, verbose=1
    )

    history = nn.fit(
        X_train_scaled, y_train_clf.astype(np.float32),
        validation_data=(X_val_scaled, y_val_clf.astype(np.float32)),
        epochs=500, batch_size=64,
        callbacks=[early_stop],
        verbose=0
    )

    best_epoch = np.argmin(history.history['val_loss']) + 1
    nn_proba = nn.predict(X_test_scaled, verbose=0).flatten()
    nn_pred = (nn_proba >= 0.5).astype(int)

    return {
        'model': nn,
        'history': history,
        'best_epoch': best_epoch,
        'predictions': nn_pred,
        'probabilities': nn_proba,
        'f1': f1_score(y_test_clf, nn_pred),
        'auc': roc_auc_score(y_test_clf, nn_proba),
        'precision': precision_score(y_test_clf, nn_pred),
        'recall': recall_score(y_test_clf, nn_pred),
    }

print("Helper functions defined.")

## 6. Train Baseline Models (Without Lag12)

In [ ]:
print("Training baseline traditional models ({} features)...".format(len(FEATURES_BASELINE)))
baseline = train_traditional_models(df_clean, FEATURES_BASELINE, train_mask, val_mask, test_mask, "Baseline")

print("  Ridge R2:     {:.4f}".format(baseline['ridge_r2']))
print("  RF R2:        {:.4f}".format(baseline['rf_r2']))
print("  Logistic F1:  {:.4f}".format(baseline['logreg_f1']))
print("  RF Clf F1:    {:.4f}".format(baseline['rf_clf_f1']))
print("  XGBoost F1:   {:.4f}".format(baseline['xgb_f1']))

In [ ]:
print("Training baseline Neural Network (Regression)...")
nn_reg_baseline = train_nn_regression(
    baseline['X_train_scaled'], baseline['y_train'],
    baseline['X_val_scaled'], baseline['y_val'],
    baseline['X_test_scaled'], baseline['y_test']
)
print("  Best epoch: {}".format(nn_reg_baseline['best_epoch']))
print("  Test R2:    {:.4f}".format(nn_reg_baseline['r2']))
print("  Test RMSE:  {:.4f}".format(nn_reg_baseline['rmse']))

In [ ]:
print("Training baseline Neural Network (Classification)...")
nn_clf_baseline = train_nn_classification(
    baseline['X_train_scaled'], baseline['y_train_clf'],
    baseline['X_val_scaled'], baseline['y_val_clf'],
    baseline['X_test_scaled'], baseline['y_test_clf']
)
print("  Best epoch:  {}".format(nn_clf_baseline['best_epoch']))
print("  Test F1:     {:.4f}".format(nn_clf_baseline['f1']))
print("  Test AUC:    {:.4f}".format(nn_clf_baseline['auc']))

## 7. Train With-Lag12 Models

In [ ]:
print("Training lag12 traditional models ({} features)...".format(len(FEATURES_LAG12)))
lag12 = train_traditional_models(df_clean, FEATURES_LAG12, train_mask, val_mask, test_mask, "With Lag12")

print("  Ridge R2:     {:.4f}  ({:+.4f})".format(lag12['ridge_r2'], lag12['ridge_r2'] - baseline['ridge_r2']))
print("  RF R2:        {:.4f}  ({:+.4f})".format(lag12['rf_r2'], lag12['rf_r2'] - baseline['rf_r2']))
print("  Logistic F1:  {:.4f}  ({:+.4f})".format(lag12['logreg_f1'], lag12['logreg_f1'] - baseline['logreg_f1']))
print("  RF Clf F1:    {:.4f}  ({:+.4f})".format(lag12['rf_clf_f1'], lag12['rf_clf_f1'] - baseline['rf_clf_f1']))
print("  XGBoost F1:   {:.4f}  ({:+.4f})".format(lag12['xgb_f1'], lag12['xgb_f1'] - baseline['xgb_f1']))

In [ ]:
print("Training lag12 Neural Network (Regression)...")
nn_reg_lag12 = train_nn_regression(
    lag12['X_train_scaled'], lag12['y_train'],
    lag12['X_val_scaled'], lag12['y_val'],
    lag12['X_test_scaled'], lag12['y_test']
)
print("  Best epoch: {}".format(nn_reg_lag12['best_epoch']))
print("  Test R2:    {:.4f}  ({:+.4f})".format(nn_reg_lag12['r2'], nn_reg_lag12['r2'] - nn_reg_baseline['r2']))
print("  Test RMSE:  {:.4f}  ({:+.4f})".format(nn_reg_lag12['rmse'], nn_reg_lag12['rmse'] - nn_reg_baseline['rmse']))

In [ ]:
print("Training lag12 Neural Network (Classification)...")
nn_clf_lag12 = train_nn_classification(
    lag12['X_train_scaled'], lag12['y_train_clf'],
    lag12['X_val_scaled'], lag12['y_val_clf'],
    lag12['X_test_scaled'], lag12['y_test_clf']
)
print("  Best epoch:  {}".format(nn_clf_lag12['best_epoch']))
print("  Test F1:     {:.4f}  ({:+.4f})".format(nn_clf_lag12['f1'], nn_clf_lag12['f1'] - nn_clf_baseline['f1']))
print("  Test AUC:    {:.4f}  ({:+.4f})".format(nn_clf_lag12['auc'], nn_clf_lag12['auc'] - nn_clf_baseline['auc']))

## 8. Training Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

curves = [
    (axes[0, 0], nn_reg_baseline['history'], nn_reg_baseline['best_epoch'],
     'Regression Baseline', 'MSE Loss'),
    (axes[0, 1], nn_reg_lag12['history'], nn_reg_lag12['best_epoch'],
     'Regression + Lag12', 'MSE Loss'),
    (axes[1, 0], nn_clf_baseline['history'], nn_clf_baseline['best_epoch'],
     'Classification Baseline', 'BCE Loss'),
    (axes[1, 1], nn_clf_lag12['history'], nn_clf_lag12['best_epoch'],
     'Classification + Lag12', 'BCE Loss'),
]

for ax, history, best_ep, title, ylabel in curves:
    epochs = range(1, len(history.history['loss']) + 1)
    ax.plot(epochs, history.history['loss'], label='Train', color='steelblue')
    ax.plot(epochs, history.history['val_loss'], label='Validation', color='#e74c3c')
    ax.axvline(best_ep, color='green', linestyle='--', alpha=0.7,
               label='Best epoch ({})'.format(best_ep))
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Head-to-Head Comparison

In [ ]:
print("=" * 90)
print("REGRESSION COMPARISON (Test Set)")
print("=" * 90)
print()
print("{:<18} {:>10} {:>12} {:>10} {:>12}".format(
    'Model', 'Base R2', 'Base RMSE', 'Lag12 R2', 'Lag12 RMSE'))
print("-" * 65)

reg_rows = [
    ('Ridge', baseline['ridge_r2'], baseline['ridge_rmse'], lag12['ridge_r2'], lag12['ridge_rmse']),
    ('Random Forest', baseline['rf_r2'], baseline['rf_rmse'], lag12['rf_r2'], lag12['rf_rmse']),
    ('Neural Network', nn_reg_baseline['r2'], nn_reg_baseline['rmse'], nn_reg_lag12['r2'], nn_reg_lag12['rmse']),
]

for name, b_r2, b_rmse, l_r2, l_rmse in reg_rows:
    print("{:<18} {:>10.4f} {:>12.4f} {:>10.4f} {:>12.4f}".format(
        name, b_r2, b_rmse, l_r2, l_rmse))

print()
print("Delta (Lag12 - Baseline):")
for name, b_r2, b_rmse, l_r2, l_rmse in reg_rows:
    print("  {:<18} R2 {:+.4f}   RMSE {:+.4f}".format(name, l_r2 - b_r2, l_rmse - b_rmse))

print()
print("=" * 90)
print("CLASSIFICATION COMPARISON (Test Set)")
print("=" * 90)
print()
print("{:<18} {:>10} {:>10} {:>10}".format('Model', 'Base F1', 'Lag12 F1', 'Delta'))
print("-" * 50)

clf_rows = [
    ('Logistic', baseline['logreg_f1'], lag12['logreg_f1']),
    ('RF Clf', baseline['rf_clf_f1'], lag12['rf_clf_f1']),
    ('XGBoost', baseline['xgb_f1'], lag12['xgb_f1']),
    ('Neural Network', nn_clf_baseline['f1'], nn_clf_lag12['f1']),
]

for name, b_f1, l_f1 in clf_rows:
    print("{:<18} {:>10.4f} {:>10.4f} {:>+10.4f}".format(name, b_f1, l_f1, l_f1 - b_f1))

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Regression R2
ax = axes[0]
models = ['Ridge', 'RF', 'NN']
base_r2 = [baseline['ridge_r2'], baseline['rf_r2'], nn_reg_baseline['r2']]
lag12_r2 = [lag12['ridge_r2'], lag12['rf_r2'], nn_reg_lag12['r2']]

x = np.arange(len(models))
width = 0.35
bars1 = ax.bar(x - width/2, base_r2, width, label='Baseline', color='steelblue', alpha=0.8)
bars2 = ax.bar(x + width/2, lag12_r2, width, label='With Lag12', color='#e74c3c', alpha=0.8)

for bar, val in zip(bars1, base_r2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            '{:.3f}'.format(val), ha='center', va='bottom', fontsize=8)
for bar, val in zip(bars2, lag12_r2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            '{:.3f}'.format(val), ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylabel('Test R$^2$')
ax.set_title('Regression: Baseline vs With Lag12')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, max(lag12_r2) * 1.15)

# Classification F1
ax = axes[1]
models_clf = ['Logistic', 'RF Clf', 'XGBoost', 'NN']
base_f1 = [baseline['logreg_f1'], baseline['rf_clf_f1'], baseline['xgb_f1'], nn_clf_baseline['f1']]
lag12_f1 = [lag12['logreg_f1'], lag12['rf_clf_f1'], lag12['xgb_f1'], nn_clf_lag12['f1']]

x = np.arange(len(models_clf))
bars1 = ax.bar(x - width/2, base_f1, width, label='Baseline', color='steelblue', alpha=0.8)
bars2 = ax.bar(x + width/2, lag12_f1, width, label='With Lag12', color='#e74c3c', alpha=0.8)

for bar, val in zip(bars1, base_f1):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            '{:.3f}'.format(val), ha='center', va='bottom', fontsize=8)
for bar, val in zip(bars2, lag12_f1):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            '{:.3f}'.format(val), ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(models_clf)
ax.set_ylabel('Test F1')
ax.set_title('Classification: Baseline vs With Lag12')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, max(lag12_f1) * 1.15)

plt.tight_layout()
plt.show()

## 10. Feature Importance

In [ ]:
# RF feature importance (with lag12)
rf_model = lag12['rf_model']
importance_df = pd.DataFrame({
    'Feature': FEATURES_LAG12,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("RF Feature Importance (With Lag12):")
print("-" * 55)
for _, row in importance_df.head(15).iterrows():
    marker = " <-- NEW" if row['Feature'] == 'delay_rate_lag12' else ""
    print("  {:<35} {:.4f}{}".format(row['Feature'], row['Importance'], marker))

lag12_rank = list(importance_df['Feature']).index('delay_rate_lag12') + 1
lag12_imp = importance_df[importance_df['Feature'] == 'delay_rate_lag12']['Importance'].values[0]
print()
print("Lag12 rank: {} (importance: {:.4f})".format(lag12_rank, lag12_imp))

In [ ]:
# NN first-layer weight analysis (with lag12)
if HAS_TF:
    nn_weights = nn_reg_lag12['model'].layers[0].get_weights()[0]  # shape: (n_features, 32)
    # Mean absolute weight per feature across all neurons
    nn_importance = np.mean(np.abs(nn_weights), axis=1)
    nn_importance_df = pd.DataFrame({
        'Feature': FEATURES_LAG12,
        'Mean_Abs_Weight': nn_importance
    }).sort_values('Mean_Abs_Weight', ascending=False)

    print("NN First-Layer Weight Magnitude (With Lag12):")
    print("-" * 55)
    for _, row in nn_importance_df.head(15).iterrows():
        marker = " <-- NEW" if row['Feature'] == 'delay_rate_lag12' else ""
        print("  {:<35} {:.4f}{}".format(row['Feature'], row['Mean_Abs_Weight'], marker))

    lag12_nn_rank = list(nn_importance_df['Feature']).index('delay_rate_lag12') + 1
    print()
    print("Lag12 NN rank: {}".format(lag12_nn_rank))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# RF importance
ax = axes[0]
top_rf = importance_df.head(12)
y_pos = np.arange(len(top_rf))
colors_rf = ['#e74c3c' if f == 'delay_rate_lag12' else 'steelblue' for f in top_rf['Feature']]
ax.barh(y_pos, top_rf['Importance'], color=colors_rf, alpha=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(top_rf['Feature'])
ax.invert_yaxis()
ax.set_xlabel('Importance')
ax.set_title('RF Feature Importance')
ax.grid(True, alpha=0.3, axis='x')

# NN weight importance
if HAS_TF:
    ax = axes[1]
    top_nn = nn_importance_df.head(12)
    y_pos = np.arange(len(top_nn))
    colors_nn = ['#e74c3c' if f == 'delay_rate_lag12' else '#9b59b6' for f in top_nn['Feature']]
    ax.barh(y_pos, top_nn['Mean_Abs_Weight'], color=colors_nn, alpha=0.8)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(top_nn['Feature'])
    ax.invert_yaxis()
    ax.set_xlabel('Mean |Weight|')
    ax.set_title('NN First-Layer Weight Magnitude')
    ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## 11. Route-Level Comparison

In [ ]:
# Generate predictions for route-level analysis
df_test = df_clean[test_mask].copy()

# Baseline NN predictions
df_test['nn_baseline_pred'] = nn_reg_baseline['predictions']
# Lag12 NN predictions
df_test['nn_lag12_pred'] = nn_reg_lag12['predictions']

# Also get RF predictions for comparison
df_test['rf_baseline_pred'] = baseline['rf_model'].predict(baseline['X_test'])
df_test['rf_lag12_pred'] = lag12['rf_model'].predict(lag12['X_test'])

print("Route-Level Regression Performance (Test Set R2):")
print("=" * 95)
print("{:<20} {:>10} {:>10} {:>10} {:>10} {:>10}".format(
    'Route', 'NN Base', 'NN Lag12', 'NN Delta', 'RF Base', 'RF Lag12'))
print("-" * 95)

route_results = []
for route in sorted(df_test['route'].unique()):
    rd = df_test[df_test['route'] == route]
    nn_b_r2 = r2_score(rd['delay_rate'], rd['nn_baseline_pred'])
    nn_l_r2 = r2_score(rd['delay_rate'], rd['nn_lag12_pred'])
    rf_b_r2 = r2_score(rd['delay_rate'], rd['rf_baseline_pred'])
    rf_l_r2 = r2_score(rd['delay_rate'], rd['rf_lag12_pred'])
    nn_delta = nn_l_r2 - nn_b_r2

    print("{:<20} {:>10.4f} {:>10.4f} {:>+10.4f} {:>10.4f} {:>10.4f}".format(
        route, nn_b_r2, nn_l_r2, nn_delta, rf_b_r2, rf_l_r2))
    route_results.append({
        'Route': route,
        'NN_Baseline': nn_b_r2, 'NN_Lag12': nn_l_r2, 'NN_Delta': nn_delta,
        'RF_Baseline': rf_b_r2, 'RF_Lag12': rf_l_r2,
    })

route_df = pd.DataFrame(route_results)
print("-" * 95)
print("{:<20} {:>10.4f} {:>10.4f} {:>+10.4f} {:>10.4f} {:>10.4f}".format(
    'OVERALL', nn_reg_baseline['r2'], nn_reg_lag12['r2'],
    nn_reg_lag12['r2'] - nn_reg_baseline['r2'],
    baseline['rf_r2'], lag12['rf_r2']))

nn_improved = (route_df['NN_Delta'] > 0).sum()
print()
print("NN improved on {} of {} routes".format(nn_improved, len(route_df)))

In [ ]:
# Route-level delta bar chart for NN
fig, ax = plt.subplots(figsize=(14, 5))

routes = route_df['Route'].values
x = np.arange(len(routes))
deltas = route_df['NN_Delta'].values
colors = ['#27ae60' if d > 0 else '#e74c3c' for d in deltas]

ax.bar(x, deltas, color=colors, alpha=0.8)
ax.axhline(y=0, color='black', linewidth=0.8)
ax.set_xticks(x)
route_labels = [r.replace('_', ' to ') for r in routes]
ax.set_xticklabels(route_labels, rotation=45, ha='right')
ax.set_ylabel('Delta R2 (NN Lag12 - NN Baseline)')
ax.set_title('Route-Level Impact of Adding Lag12 to Neural Network')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 12. NN vs Best Traditional Model (With Lag12)

In [ ]:
print("=" * 80)
print("NEURAL NETWORK vs TRADITIONAL MODELS (With Lag12)")
print("=" * 80)
print()

# Best traditional regression = RF
best_trad_r2 = max(lag12['ridge_r2'], lag12['rf_r2'])
best_trad_name_reg = 'Ridge' if lag12['ridge_r2'] > lag12['rf_r2'] else 'Random Forest'

# Best traditional classification = XGBoost
best_trad_f1 = max(lag12['logreg_f1'], lag12['rf_clf_f1'], lag12['xgb_f1'])
clf_names = ['Logistic', 'RF Clf', 'XGBoost']
clf_vals = [lag12['logreg_f1'], lag12['rf_clf_f1'], lag12['xgb_f1']]
best_trad_name_clf = clf_names[np.argmax(clf_vals)]

nn_vs_trad_r2 = nn_reg_lag12['r2'] - best_trad_r2
nn_vs_trad_f1 = nn_clf_lag12['f1'] - best_trad_f1

print("REGRESSION (with lag12):")
print("  NN R2:                  {:.4f}".format(nn_reg_lag12['r2']))
print("  Best traditional ({:<4}): {:.4f}".format(best_trad_name_reg[:4], best_trad_r2))
print("  NN advantage:           {:+.4f}".format(nn_vs_trad_r2))
print()
print("CLASSIFICATION (with lag12):")
print("  NN F1:                  {:.4f}".format(nn_clf_lag12['f1']))
print("  Best traditional ({:<4}): {:.4f}".format(best_trad_name_clf[:4], best_trad_f1))
print("  NN advantage:           {:+.4f}".format(nn_vs_trad_f1))

## 13. Summary and Observations

In [ ]:
print("=" * 80)
print("CONCLUSIONS")
print("=" * 80)
print()

# Lag12 impact on NN
nn_r2_delta = nn_reg_lag12['r2'] - nn_reg_baseline['r2']
nn_rmse_delta = nn_reg_lag12['rmse'] - nn_reg_baseline['rmse']
nn_f1_delta = nn_clf_lag12['f1'] - nn_clf_baseline['f1']

print("1. LAG12 IMPACT ON NEURAL NETWORK:")
print("   R2:   {:+.4f}  ({:.4f} -> {:.4f})".format(
    nn_r2_delta, nn_reg_baseline['r2'], nn_reg_lag12['r2']))
print("   RMSE: {:+.4f}  ({:.4f} -> {:.4f})".format(
    nn_rmse_delta, nn_reg_baseline['rmse'], nn_reg_lag12['rmse']))
print("   F1:   {:+.4f}  ({:.4f} -> {:.4f})".format(
    nn_f1_delta, nn_clf_baseline['f1'], nn_clf_lag12['f1']))
print()

print("2. LAG12 IMPACT ON TRADITIONAL MODELS:")
print("   Ridge R2:   {:+.4f}".format(lag12['ridge_r2'] - baseline['ridge_r2']))
print("   RF R2:      {:+.4f}".format(lag12['rf_r2'] - baseline['rf_r2']))
print("   XGBoost F1: {:+.4f}".format(lag12['xgb_f1'] - baseline['xgb_f1']))
print()

print("3. NN vs BEST TRADITIONAL (with lag12):")
print("   R2: NN {:.4f} vs {} {:.4f}  ({:+.4f})".format(
    nn_reg_lag12['r2'], best_trad_name_reg, best_trad_r2, nn_vs_trad_r2))
print("   F1: NN {:.4f} vs {} {:.4f}  ({:+.4f})".format(
    nn_clf_lag12['f1'], best_trad_name_clf, best_trad_f1, nn_vs_trad_f1))
print()

print("4. ROUTE-LEVEL: NN improved on {} of {} routes with lag12".format(
    nn_improved, len(route_df)))
print()

### Observations

- Including 12-month lagged delay rate improves neural network performance (R2 increased: 0.5025 -> 0.5300), which is consistent with improvements observed in the other models (see notebooks [15a](/notebooks/15a_forecasting_lag12.ipynb) and [16](/notebooks/16_nowcasting_lag12.ipynb)).
- Even with the addition of `delay_rate_lag12`, the neural network performs comparably to simpler models with lag12.
   - Considering the added complexity and reduced interpretability of a neural network model, simpler models (Ridge for regression, XGBoost for classification) are still the overall preferred models.


## 14. Next Step

The next and final step (for now) will explore the ultimate neural network for time-series prediction: Recurrent Neural Network!